# 06 — Work Item Orchestrator full loop (gated, expensive)

One end-to-end WIO cycle. The orchestrator is a single `ClaudeSDKClient` session that does:

1. **Step 1 (Phase 5, skill 10)** — Knowledge Store enrichment.
2. **Steps 2–4 (Phase 3)** — Builder → Git → QA cycle (cycle cap = 3 enforced by Pydantic + a Layer-2 escalation in this module).
3. **Step 5 (Phase 5, skill 11)** — Knowledge Store ingestion evaluation.

**This notebook can spend real money and write to real Linear / GitHub.** Default state = explore only. Three gates must all be satisfied for the loop to run:

- `HSB_NOTEBOOK_RUN_LIVE=1`
- `HSB_NOTEBOOK_WIO_TASK_ID` set to a sandbox Linear LIN-ID
- `CLAUDE_CODE_OAUTH_TOKEN` set; `ANTHROPIC_API_KEY` UNSET

Inspection-only cells (system-prompt assembly, allow-list, G3 wiring) run regardless and are the high-value half if you don't have credentials handy.

In [ ]:
import asyncio
import os
from pathlib import Path

from _helpers import assert_g1_safe, ensure_src_on_path, gated, live_mode

ROOT = ensure_src_on_path()
from hsb.agents import work_item_orchestrator as wio

## System-prompt assembly — what the LLM actually receives

`assemble_system_prompt()` reads the 7 skill files (5 lifecycle skills + skills 10/11) and concatenates them with `# SKILL: <stem>` separators. We render the size + first 200 chars of each section so you can spot a missing or duplicated skill at a glance.

In [ ]:
import re

prompt = wio.assemble_system_prompt()
print(f"total prompt size: {len(prompt):,} chars (~{len(prompt) / 4:.0f} tokens)")
for m in re.finditer(r"^# SKILL: (.+?)$", prompt, re.MULTILINE):
    print(f"  • {m.group(1)}")
print("\nSKILL_FILES:")
for f in wio.SKILL_FILES:
    print("  ", f)

## G3 wiring — runtime backstop is called on every received message

`assert_no_task_dispatch` should appear in every receive loop. We grep the source for the call sites — if any loop loses the call, the runtime backstop disappears.

In [ ]:
src = Path(wio.__file__).read_text()
calls = src.count("assert_no_task_dispatch(")
print(f"assert_no_task_dispatch call sites: {calls}")
assert calls >= 3, (
    f"G3 runtime backstop missing or reduced (found {calls}, expected >= 3)"
)
# WORC-02 structural defense: no Agent in allowed_tools, no agents= kwarg.
assert '"Agent"' not in src, "WIO allow-list contains Agent"
assert "AgentDefinition" not in src, (
    "WIO imports AgentDefinition (sub-subagent dispatch surface)"
)
assert "agents=" not in src.replace("agents={", "").replace("agents=[", ""), (
    "WIO uses agents= kwarg"
)
print("WIO: WORC-02 structural defenses intact")

## In-process MCP tool wrappers (`@tool`)

Phase 2 agents are exposed via `create_sdk_mcp_server` rather than as sub-agents. The wrappers must each return the canonical `{"content":[{"type":"text","text":...}]}` envelope (Pitfall 4) — returning a Pydantic model directly silently fails the SDK serializer.

In [ ]:
for name in ["run_linear_tool", "run_builder_tool", "run_git_tool", "run_qa_tool"]:
    fn = getattr(wio, name, None)
    assert fn is not None, f"WIO missing {name}"
# Spot-check one wrapper for the canonical envelope shape in source.
for name in ["run_linear_tool", "run_builder_tool", "run_git_tool", "run_qa_tool"]:
    block = src[src.index(f"async def {name}") : src.index(f"async def {name}") + 2000]
    assert '"content"' in block and '"text"' in block, (
        f"{name} missing canonical envelope"
    )
print("WIO: 4 @tool wrappers present, all return canonical content envelope")

## Live run (gated)

If all three gates are satisfied, drive a single task end-to-end. Output is structured but verbose (skill prompts + every tool call); expect a few minutes wall-clock and several hundred KB of stdout.

**Before clicking run:** confirm the LIN-ID points to a sandbox project, not a production board.

In [ ]:
task_id = os.environ.get("HSB_NOTEBOOK_WIO_TASK_ID")
if not live_mode() or not task_id:
    print(
        gated(
            "WIO live run (set HSB_NOTEBOOK_WIO_TASK_ID=LIN-... and HSB_NOTEBOOK_RUN_LIVE=1)"
        )
    )
else:
    assert_g1_safe()
    print(f"Running WIO cycle for {task_id}…")
    asyncio.run(wio.run_orchestration_cycle(work_item_id=task_id))
    print("cycle complete — inspect Linear comments + GitHub PR for the artifacts")